In [9]:
import torch, transformers
print(torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print(transformers.__version__)   # the models were trained with 4.49, so use 4.49 or newer

2 ['Tesla T4', 'Tesla T4']
5.20.2


In [10]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

REPOS = {
    "hvs-3b-sft": ("humanoid-vstar/HVS-3B-sft-only", "cuda:0"),
    "hvs-3b":     ("humanoid-vstar/HVS-3B",          "cuda:1"),
}
MODELS = {}
for key, (repo, dev) in REPOS.items():
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        repo, torch_dtype=torch.float16, attn_implementation="sdpa"
    ).to(dev).eval()
    proc = AutoProcessor.from_pretrained(repo, max_pixels=448*448)  # caps image tokens
    MODELS[key] = (model, proc)
    print(key, "loaded on", dev)

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

hvs-3b-sft loaded on cuda:0


Loading weights:   0%|          | 0/825 [00:00<?, ?it/s]

hvs-3b loaded on cuda:1


In [11]:
def generate(key, messages, images, max_new_tokens=512):
    model, proc = MODELS[key]
    text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[text], images=images, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

In [12]:
from huggingface_hub import HfApi, hf_hub_download
files = HfApi().list_repo_files("humanoid-vstar/hstar_bench_png", repo_type="dataset")
print(len(files)); print(files[:30])

4
['.gitattributes', 'README.md', 'hos_bench.zip', 'hps_bench.zip']


In [15]:
import numpy as np, cv2

def render_view(pano, yaw, pitch, fov=90, size=448):
    H, W = pano.shape[:2]
    f = size / 2 / np.tan(np.radians(fov) / 2)
    i, j = np.meshgrid(np.arange(size), np.arange(size))
    x = (i + 0.5 - size / 2) / f
    y = -(j + 0.5 - size / 2) / f
    ya, pa = np.radians(yaw), np.radians(pitch)
    cy, sy, cp, sp = np.cos(ya), np.sin(ya), np.cos(pa), np.sin(pa)
    fwd, right, up = np.array([sy*cp, sp, cy*cp]), np.array([cy, 0, -sy]), np.array([-sy*sp, cp, -cy*sp])
    d = fwd + x[..., None] * right + y[..., None] * up
    lon = np.arctan2(d[..., 0], d[..., 2])
    lat = np.arctan2(d[..., 1], np.hypot(d[..., 0], d[..., 2]))
    u = ((lon / (2*np.pi) + 0.5) * W).astype(np.float32)
    v = np.clip((0.5 - lat / np.pi) * H, 0, H - 1).astype(np.float32)
    return cv2.remap(pano, u, v, cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)

In [16]:
import glob, time, re
import numpy as np
from PIL import Image

# 1) Load the panorama and render ONE perspective view (this defines pil_image)
path = glob.glob("/kaggle/input/datasets/ghassenayari/360image/360.jpg", recursive=True)
print("found:", path)
pano = np.array(Image.open(path[0]).convert("RGB"))
pil_image = Image.fromarray(render_view(pano, 101, -6))   # looking toward the red bus
pil_image.save("/kaggle/working/view_test.jpg")           # open it to check it looks normal

# 2) Prompt: replace PROMPT with the official template once you have it
PROMPT = """You are a humanoid agent standing in a 360-degree scene and can only see a narrow view.
Yaw is in [0,360) clockwise, pitch in [-90,90] positive up.
Think inside <think></think>, then output one action inside <answer></answer>:
rotate(dyaw,dpitch) or submit(yaw,pitch).
[View 1] viewing direction: (101,-6)
Human Instruction: look for the red and white bus
Decide your next action."""

messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": PROMPT},
]}]

# 3) Run both models and check the output format
for key in MODELS:
    t = time.time()
    out = generate(key, messages, [pil_image])
    secs = time.time() - t
    has_think = "<think>" in out
    has_action = bool(re.search(r"(rotate|submit)\s*\(", out, re.I))
    print(f"\n=== {key} | {secs:.1f}s | think={has_think} | action={has_action}")
    print(out)

found: ['/kaggle/input/datasets/ghassenayari/360image/360.jpg']


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(



=== hvs-3b-sft | 17.1s | think=True | action=True
<think>The current view shows a red and white bus on a stone bridge, with people around it. The bus is clearly visible and appears to be the main focus of the image. Since the task is to look for the red and white bus, and it is already in view, there is no need to rotate further. Therefore, the next action is to submit(101,-6).</think><answer>submit(101,-6)</answer>

=== hvs-3b | 18.8s | think=True | action=True
<think>The current view shows a red and white bus on a stone bridge, with a tall building in the background. The bus is prominently visible and appears to be the main focus of the image. Since the task is to look for the red and white bus, and it is clearly in view, it makes sense to submit the current position. Therefore, the next action is submit(101,-6).</think><answer>submit(101,-6)</answer>
